#  **Data Collection and Preprocessing**

### Import Libraries

In [1]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

In [ ]:
import pandas as pd
import numpy as np
from google_play_scraper import app, Sort, reviews
import re
from datetime import datetime
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    preprocessing_report,
    save_cleaned_data,
)
from src.data_scrapping import scrap_reviews

### Web Scraping

#### App metadata

In [3]:
BOA_APP_ID = 'com.boa.boaMobileBanking'
display_app_info(BOA_APP_ID)

BoA Mobile App Info
App Title   : BoA Mobile
Current Score: 4.3922596
Total Ratings: 9,219
Total Reviews: 1,461
Installs     : 1,000,000+


#### Scrape reviews

In [4]:
reviews = scrap_reviews(app_id=BOA_APP_ID)

Scraping reviews for com.boa.boaMobileBanking...
Collected 700 raw reviews


#### Collect review text, rating, review date, bank , source

In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: c3bb042c-844b-4580-98b9-df418622b2fb
  userName: Hamid Abdella
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocKxYO7gSAW0PgoFvwbHi4RxuoH8jHuTKOA20Pv-CtODfkn93w=mo
  content: it's very good app
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: None
  at: 2026-05-12 11:50:32
  replyContent: None
  repliedAt: None
  appVersion: None


In [6]:
df = review_dataframe(reviews, app_info={'title': 'BOA Bank'})

print(f"Shape: {df.shape}")
df.head()

Shape: (700, 6)


,review_id,review,rating,date,bank,source
0,c3bb042c-844b-4580-98b9-df418622b2fb,it's very good app,5,2026-05-12 11:50:32,BOA Bank,Google Play
1,400ce769-3726-43b2-ac4d-755b3a15f026,this app is good but the speed of app is very ...,2,2026-05-11 18:18:54,BOA Bank,Google Play
2,4d6d2f22-5e71-47be-9cde-a1cf6c9fff93,good,5,2026-05-09 14:41:50,BOA Bank,Google Play
3,e77089b3-aecf-45e2-a64a-ce917fc4233a,boa the best,5,2026-05-08 13:47:07,BOA Bank,Google Play
4,41c64c67-b81a-4326-83c4-e95044aef7f6,bank of absiniya is best bank in ethiopian,5,2026-05-07 10:33:06,BOA Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [7]:
df_clean = df.copy()

In [8]:
df_clean = remove_duplicates(df_clean)

Removed 0 duplicate reviews
Remaining: 700 reviews


#### Handle missing values

In [9]:
df_clean =handle_missing_data(df_clean)

Removed 0 rows with missing critical data
Remaining: 700 reviews


#### Normalize dates to YYYY-MM-DD format

In [10]:
df_clean = normalize_dates(df_clean)

Before normalization:
0   2026-05-12 11:50:32
1   2026-05-11 18:18:54
2   2026-05-09 14:41:50
dtype: datetime64[us]

After normalization:
0    2026-05-12
1    2026-05-11
2    2026-05-09
dtype: str

Date range: 2024-08-19 to 2026-05-12


#### Handle incorrect ratings

In [11]:
df_clean = validate_rating(df_clean)

All ratings are valid (1-5).
Remaining: 700 reviews


#### Clean review text

In [12]:
df['review'] = df['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df['review'].head(10).to_string())

Sample cleaned reviews:
0                                   it's very good app
1    this app is good but the speed of app is very ...
2                                                 good
3                                         boa the best
4           bank of absiniya is best bank in ethiopian
5                                                     
6                                                 good
7    extremely slow app and unreliable for most pay...
8                                          amazing app
9    i tried to oppen mobile app of boa, but it can...


#### Save the cleaned dataset

In [13]:
# Select only the 5 required columns in the right order
df_clean = df_clean[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (700, 5)


,review,rating,date,bank,source
0,it's very good app,5,2026-05-12,BOA Bank,Google Play
1,this app is good but the speed of app is very ...,2,2026-05-11,BOA Bank,Google Play
2,good,5,2026-05-09,BOA Bank,Google Play
3,boa the best,5,2026-05-08,BOA Bank,Google Play
4,bank of absiniya is best bank in ethiopian,5,2026-05-07,BOA Bank,Google Play
5,አስተማማኝና ዘመኑን የዋጀ,4,2026-05-05,BOA Bank,Google Play
6,good,5,2026-05-04,BOA Bank,Google Play
7,extremely slow app and unreliable for most pay...,2,2026-05-03,BOA Bank,Google Play
8,Amazing app,5,2026-05-02,BOA Bank,Google Play
9,"I tried to oppen mobile app of BOA, but it can...",1,2026-05-01,BOA Bank,Google Play


In [14]:
save_cleaned_data(df_clean, output_path="../data/processed/boa_reviews_cleaned.csv")

Cleaned data saved to ../data/processed/boa_reviews_cleaned.csv
Saved to: ../data/processed/boa_reviews_cleaned.csv


### Report

In [15]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    700
  Reviews after cleaning :    700
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2024-08-19  to  2026-05-12
Rating distribution:
  5 stars:  373  ██████████████████████████████████████████████████████████████████████████
  4 stars:   48  █████████
  3 stars:   35  ███████
  2 stars:   22  ████
  1 stars:  222  ████████████████████████████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 16 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

